# COGS 188 - Project Proposal

## Learning to Play 11×11 Hex: Self-Play Neural Networks vs. Classical Search

# Names

- Anshul Kolli
- Brycen Martin
- Cameron Chen

# Abstract
Our goal is to build an agent that plays the board game Hex on the standard 11×11 board, a game with 121 possible opening moves, roughly $10^{57}$ possible positions, and no known winning strategy at this size. Our data will be generated by the agents themselves: every self-play game produces a sequence of board positions, each stored as a 3×11×11 tensor together with the move probabilities found by tree search and the final game result. We will compare four agents of increasing sophistication: a random player, alpha-beta minimax with a shortest-path evaluation function, Monte Carlo Tree Search (MCTS) with random rollouts, and an AlphaZero-style MCTS guided by a policy-value neural network trained through self-play. We will tune the key hyperparameters of the strongest agents (simulations per move, exploration constant, network size, learning rate) across multiple random seeds. Success will be measured by head-to-head win rates with 95% confidence intervals (each agent plays both colors equally), Elo ratings fitted across all agents, the number of search simulations an agent needs to match another agent's strength, and average time per move.

# Background

Hex is a two-player connection game invented by Piet Hein in 1942 and independently by John Nash in 1949<a name="hayward"></a>[<sup>[1]</sup>](#haywardnote). Players take turns placing a stone of their color on an empty cell of a rhombus-shaped board of hexagons; Red tries to connect the top and bottom edges and Blue tries to connect the left and right edges. A game of Hex can never end in a draw, a fact that is mathematically equivalent to the Brouwer fixed-point theorem<a name="gale"></a>[<sup>[2]</sup>](#galenote). A strategy-stealing argument shows that the first player always has a winning strategy, but the argument does not say what that strategy is, and deciding the winner of a general Hex position is PSPACE-complete<a name="reisch"></a>[<sup>[3]</sup>](#reischnote). Small boards have been solved by computer search (8×8 by Henderson et al.<a name="henderson"></a>[<sup>[4]</sup>](#hendersonnote) and 9×9 by Pawlewicz and Hayward<a name="pawlewicz"></a>[<sup>[5]</sup>](#pawlewicznote)), but the standard 11×11 board remains unsolved. Its 121-move branching factor at the start of the game is far larger than chess, and its rules are simple enough to implement and simulate quickly. That combination makes Hex a common research testbed for game-playing AI.

Early strong Hex programs used classical game-tree search. Anshelevich's program Hexy combined alpha-beta search with an evaluation function that models the board as an electrical circuit and with "virtual connections," patterns of stones that are guaranteed to connect<a name="anshelevich"></a>[<sup>[6]</sup>](#anshelevichnote). These approaches depend heavily on hand-designed evaluation functions. Monte Carlo Tree Search (MCTS) removed much of that dependence: the UCT algorithm treats move selection at each tree node as a multi-armed bandit problem and estimates position values by simulating random games to the end<a name="kocsis"></a>[<sup>[7]</sup>](#kocsisnote)<a name="browne"></a>[<sup>[8]</sup>](#brownenote). Random playouts are especially well suited to Hex because a randomly filled board always has exactly one winner. The MCTS-based program MoHex won the Computer Olympiad Hex tournament in 2009, 2010, and 2011<a name="arneson"></a>[<sup>[9]</sup>](#arnesonnote).

The most recent advance replaces random playouts and hand-tuned heuristics with neural networks trained through self-play. AlphaGo Zero and AlphaZero showed that a single network predicting both a move distribution (policy) and an expected outcome (value) can guide MCTS to superhuman strength in Go, chess, and shogi without human game data<a name="silver2017"></a>[<sup>[10]</sup>](#silver2017note)<a name="silver2018"></a>[<sup>[11]</sup>](#silver2018note). Around the same time, Anthony et al. introduced Expert Iteration, a closely related method, and showed on 9×9 Hex that an agent trained entirely through self-play could defeat MoHex 1.0, the most recent publicly released version of the Olympiad champion<a name="anthony"></a>[<sup>[12]</sup>](#anthonynote). Gao et al. later extended this line of work on Hex with a three-head network that also predicts action values, and the resulting program, MoHex-3HNN, won the 2018 Computer Olympiad Hex tournament<a name="gao"></a>[<sup>[13]</sup>](#gaonote).

These systems were trained with very large compute budgets, which leaves a practical question open: **how much does a learned policy-value network actually help tree search when compute is limited to what a student team has access to?** Answering this lets us directly compare three families of methods from this course (heuristic minimax search, Monte Carlo planning, and reinforcement learning through self-play) on the same nontrivial problem under equal search budgets.

# Problem Statement

We will build and compare agents that play 11×11 Hex, and measure how much a self-play-trained policy-value network improves Monte Carlo Tree Search under a fixed compute budget.

**Formal definition.** Hex is a two-player, zero-sum, deterministic game with perfect information. A state $s$ consists of the board $b \in \{\text{empty}, \text{Red}, \text{Blue}\}^{121}$ and the player to move. The legal actions $A(s)$ are the empty cells, so $|A(s)| \le 121$. The game ends when one player connects their two edges, giving reward $+1$ to the winner and $-1$ to the loser; draws are impossible. The number of board configurations is bounded by $3^{121} \approx 5.4 \times 10^{57}$, so exhaustive search is infeasible and agents must generalize from search and experience.

An agent is a policy $\pi(a \mid s)$ that may use search. For a fixed search budget of $N$ simulations per move, we want to find the agent and hyperparameter setting that maximizes expected outcome against a fixed pool of opponent agents, with both colors played equally.

**Specific questions:**
1. At equal simulations per move, does MCTS guided by a policy-value network (PV-MCTS) achieve a higher win rate than MCTS with random rollouts (UCT) and than alpha-beta search?
2. How many simulations does UCT need to match PV-MCTS's strength at a given budget? This measures how much search the learned network saves.
3. Which hyperparameters (simulations per move, exploration constant, network size, learning rate) most affect playing strength, and what is the simplest configuration that performs as well as the best one?

**Why it is quantifiable, measurable, and replicable.** Every game ends in a win or loss, so playing strength reduces to win rates and Elo ratings with well-defined confidence intervals. Every experiment is a match between two fixed agents started from controlled random seeds and a fixed set of opening positions, so any match can be rerun exactly and repeated as many times as needed.

# Data

Hex does not need an external dataset, because the agents generate their own data through simulated games. We will produce two datasets.

**1. Self-play training data (used to train the policy-value network)**
- **Source:** Generated by our own Hex engine, written in Python and NumPy, and checked against the `hex` game in DeepMind's OpenSpiel library<a name="lanctot"></a>[<sup>[14]</sup>](#lanctotnote) (`pyspiel.load_game("hex(board_size=11)")`) to confirm that both produce identical legal moves and winners.
- **Size:** We plan at least 3,000 self-play games. 11×11 games typically last 60 to 90 moves, so this gives roughly 200,000 positions, or about 400,000 after symmetry augmentation. That is far above the 10k-observation guideline.
- **What an observation is:** One board position from one self-play game, stored as:
  - **Input `x`:** a 3×11×11 binary tensor (363 features): the current player's stones, the opponent's stones, and the empty cells.
  - **Policy target `π`:** a 121-dimensional probability vector, equal to the MCTS visit counts at that position normalized to sum to 1.
  - **Value target `z`:** $+1$ if the player to move at that position eventually won the game, $-1$ otherwise.
  - **Metadata:** game ID, move number, training iteration, and random seed.
- **Special handling:**
  - **Canonical perspective:** Positions are always stored from the point of view of the player to move. When Blue is to move, we reflect the board across its long diagonal and swap colors, so the network always "plays Red" (connecting top to bottom). This is valid because that reflection maps one player's goal onto the other's.
  - **Augmentation:** Each position is also stored rotated 180°, which leaves the game unchanged and doubles the data.
  - **Replay buffer:** Training samples come from the most recent games only (a sliding window), so the network learns from positions produced by recent versions of itself.
  - **Train/validation split by game:** Validation positions come from whole held-out games rather than randomly chosen positions, because positions from the same game are highly correlated.

**2. Evaluation match records (used to measure performance)**
- Every evaluation game is logged with: the two agents and their settings, which agent played first, the opening position, the full move list, the winner, and the time per move.
- Thousands of games are planned across all agent pairings and hyperparameter settings. One row per game, with more than 10 variables per row.
- **First-move advantage:** The first player has a large advantage in Hex, so every pairing is played from a fixed set of opening positions, and each opening is played twice with the agents switching colors. This cancels out the first-player advantage when comparing agents.

# Proposed Solution

We will implement four agents, each a stronger version of the one before it. Agents 0 to 2 serve as benchmarks for Agent 3, which is the main solution.

**Game engine.** A Python/NumPy Hex implementation that detects wins with a union-find data structure (merging connected groups of stones as moves are played), so checking for a winner takes near-constant time per move. We will unit-test it on hand-built positions and compare it against OpenSpiel's Hex on thousands of random games.

**Agent 0: Random.** Picks a uniformly random legal move. This is the floor that every other agent must clearly beat.

**Agent 1: Alpha-beta minimax (classical benchmark).** Depth-limited minimax with alpha-beta pruning and iterative deepening under a per-move time limit. Positions are scored with a shortest-path heuristic: using Dijkstra's algorithm, compute how many more stones each player needs to connect their edges (own stones cost 0, empty cells cost 1, opponent stones are blocked), and use the difference between the two players' counts. Moves are ordered by this heuristic to improve pruning.

**Agent 2: UCT Monte Carlo Tree Search (main benchmark).** Standard MCTS<a name="kocsis2"></a>[<sup>[7]</sup>](#kocsisnote) using UCB1 to choose moves within the tree:
$$a^* = \arg\max_a \left[ Q(s,a) + c \sqrt{\frac{\ln N(s)}{N(s,a)}} \right]$$
where $Q(s,a)$ is the average result after playing $a$, $N(s)$ is the number of visits to $s$, and $c$ is the exploration constant. Leaf positions are evaluated with a random playout: fill the remaining empty cells randomly and check who connected. This is the strongest method that does not use learning.

**Agent 3: Policy-value MCTS trained by self-play (our solution).** Following AlphaZero<a name="silver2018b"></a>[<sup>[11]</sup>](#silver2018note) and Expert Iteration<a name="anthony2"></a>[<sup>[12]</sup>](#anthonynote):
- **Network:** A residual convolutional network $f_\theta(s) = (\mathbf{p}, v)$ implemented in PyTorch. Starting size is 4 residual blocks with 64 channels. It outputs a probability for each of the 121 cells (with illegal moves masked out) and a value $v \in [-1, 1]$ predicting the winner.
- **Search:** MCTS with the PUCT selection rule, which uses the network's move probabilities $P(s,a)$ to focus the search:
$$a^* = \arg\max_a \left[ Q(s,a) + c_{\text{puct}}\, P(s,a) \frac{\sqrt{N(s)}}{1 + N(s,a)} \right]$$
  Leaves are evaluated with the network's value $v$ instead of a random playout.
- **Self-play training loop:**
  1. Play self-play games with the current network. Add Dirichlet noise to the root move probabilities and sample moves with temperature $\tau = 1$ for the first ~10 moves to encourage exploration, then play the most-visited move.
  2. Store each position with its MCTS visit distribution $\pi$ and the final result $z$.
  3. Train the network on samples from the replay buffer by minimizing
  $$\ell = (z - v)^2 - \pi^\top \log \mathbf{p} + \lambda \lVert\theta\rVert^2$$
  4. Keep the new network only if it beats the previous one in an evaluation match (win rate > 55%), then repeat.
- **Compute:** Training runs on our laptops and Google Colab GPUs. Our network and simulation budgets (50 to 800 simulations per move) are deliberately small so that a full training run takes hours rather than weeks, which leaves room for multiple seeds.

**Model selection.** We will tune hyperparameters with a coarse grid search, using 3 random seeds for every setting that involves training:

| Agent | Hyperparameter | Values tried |
|---|---|---|
| UCT and PV-MCTS | Simulations per move | 50, 100, 200, 400, 800 |
| UCT | Exploration constant $c$ | 0.5, 1.0, 1.4, 2.0 |
| PV-MCTS | Exploration constant $c_{\text{puct}}$ | 0.5, 1, 2, 4 |
| PV-MCTS | Network size (blocks × channels) | 2×32, 4×64, 6×64 |
| PV-MCTS | Learning rate | $10^{-3}$, $10^{-4}$ |
| Alpha-beta | Search depth or time per move | 2, 3, 4 ply |

Configurations are ranked by Elo against the fixed opponent pool. Among configurations whose Elo confidence intervals overlap the best one, we will pick the simplest (fewest parameters, fewest simulations), following the parsimony principle.

**How the solution will be tested:**
- **Engine correctness:** unit tests for move legality and win detection, plus agreement with OpenSpiel on random games.
- **Search sanity checks:** every search agent must find the winning move in hand-built "win in one" and "block or lose" positions.
- **Final comparison:** a round-robin tournament among the best configuration of each agent, with paired openings and color swaps. Pairwise differences will be tested with a two-sided binomial test against a 50% win rate ($\alpha = 0.05$).
- **Reproducibility:** all random seeds, configuration files, network checkpoints, and match logs will be committed to our GitHub repository.

**Stretch goal:** If time permits, we will play our best agent against MoHex, the open-source champion Hex program from the University of Alberta's Benzene project<a name="arneson2"></a>[<sup>[9]</sup>](#arnesonnote), at a matched time per move.

# Evaluation Metrics

Every metric below applies equally to the benchmark agents (random, alpha-beta, UCT) and to our solution (PV-MCTS).

**1. Head-to-head win rate with a 95% confidence interval (primary).** If agent A wins $W$ of $n$ games against agent B (half with each color), its win rate is $\hat{p} = W/n$. We report the Wilson score interval<a name="wilson"></a>[<sup>[15]</sup>](#wilsonnote), which stays accurate when win rates are near 0% or 100%:
$$\frac{\hat{p} + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}}{1 + \frac{z^2}{n}}, \quad z = 1.96$$
For example, 130 wins out of 200 games gives $\hat{p} = 0.65$ with a 95% interval of about $[0.58, 0.71]$, so A is significantly stronger than B.

**2. Elo rating.** Win rates only compare two agents at a time; Elo<a name="elo"></a>[<sup>[16]</sup>](#elonote) places all agents and configurations on one scale. Under the Elo model, A's expected score against B is
$$E_A = \frac{1}{1 + 10^{(R_B - R_A)/400}}$$
We fit all ratings by maximum likelihood over every recorded game, with the random agent fixed at 0, and estimate confidence intervals by bootstrapping (resampling games). A 100-point gap corresponds to an expected win rate of about 64%.

**3. Simulation efficiency.** This directly answers how much search the learned network saves. For PV-MCTS using $N$ simulations per move, find the smallest UCT budget $N^*$ at which UCT wins at least 50% against it. We report the ratio $N^*/N$; a ratio of 8, for example, means the network does the work of 8 times more random-playout search.

**4. Time per move.** The average wall-clock time per move (including network inference) on the same hardware. A network that is stronger per simulation but much slower per simulation may not be better overall, so we compare agents both at equal simulations and at equal time.

**5. Network quality (diagnostic for PV-MCTS only).** Measured on held-out validation games:
- **Value error:** mean squared error $\frac{1}{m}\sum_i (z_i - v_i)^2$ between predicted and actual game results.
- **Policy accuracy:** how often the network's highest-probability move matches the move MCTS visited most.

These track whether more training is still helping and give the learning curves for the final report.

# Ethics & Privacy

Hex has fewer direct harms than projects involving people's data, but we identified several concerns, using the Deon checklist (https://deon.drivendata.org) as a guide:

- **Data privacy:** All of our data is generated by our own agents, so no personal information is involved. If we later use human game records (for example, for an optional comparison), we will use only public archives, respect their terms of use, and strip player usernames.
- **Cheating in online play:** A strong Hex agent could be used to cheat on online Hex sites where people play rated games. We will not connect our agent to any site where people play rated games, and our repository README will state that the agent should not be used in human competition.
- **Compute and energy use:** Self-play training can use a lot of energy. We will keep training budgets modest, record GPU-hours for each experiment, and report them in the final report. Our focus on small compute budgets also makes the results more useful to groups without large-scale compute.
- **Overstating results:** Game results are easy to cherry-pick. We will report every configuration we tried (not just the best one), include confidence intervals on every comparison, compare agents at equal compute, and avoid claiming that results on Hex generalize to real-world decision-making without evidence.
- **Attribution and licensing:** We will credit any code we reuse (OpenSpiel is Apache 2.0 licensed) and the published methods we implement.

# Team Expectations 

* **Communication:** We use a group chat for day-to-day communication and respond within 24 hours. We meet in person or on Zoom at least once a week, and anyone who can't make it reads the meeting notes and catches up by the next day.
* **Dividing the work:** Each major piece of the project has one owner (see the timeline), but nothing is one person's job alone. Every piece of code is reviewed by at least one other member before it is merged.
* **Code and notebooks:** All work goes in our GitHub repository. We work on separate branches and merge through pull requests so nobody overwrites anyone else's work. We clear outputs from large notebooks before committing, and long training runs log results to files instead of notebook output.
* **Shared compute:** Training runs take hours, so we post in the group chat before starting a long run and share its results and checkpoints when it finishes.
* **Deadlines:** Our internal deadline is 48 hours before each real deadline, so there is time to review and fix things. If someone is falling behind or stuck, they tell the group as early as possible rather than the night before; asking for help is expected.
* **Decisions:** We discuss decisions together and aim for consensus. If we can't agree, we take a majority vote, which always resolves with three members.
* **Conflict:** Disagreements are raised directly and respectfully with the person involved first, then with the whole group. If an issue such as a persistent imbalance in effort can't be resolved within the group, we will bring it to the TA.
* **Effort:** Everyone contributes roughly equally over the quarter. Busy weeks happen, so workloads can shift as long as they balance out.

# Project Timeline Proposal

Dates are for Fall 2026; we will adjust them to match the official course deadlines.

| Meeting Date | Meeting Time | Completed Before Meeting | Discuss at Meeting |
|---|---|---|---|
| 11/5 | 7 PM | Read the AlphaZero and Expert Iteration papers (all) | Finalize proposal; assign sections; set up GitHub repo and Python environment |
| 11/7 | 7 PM | Write and finalize proposal sections (all); submit proposal (Anshul) | Agree on code structure and interfaces between engine, agents, and match runner |
| 11/12 | 7 PM | Hex engine with union-find and unit tests, checked against OpenSpiel (Brycen); random and alpha-beta agents (Cameron); UCT MCTS agent (Anshul) | Review engine and baseline code; run first baseline matches; plan the network |
| 11/19 | 7 PM | Policy-value network and self-play training loop (Anshul and Cameron); match runner with paired openings, Wilson intervals, and Elo fitting (Brycen) | Inspect first training run and learning curves; debug |
| 11/24 | 7 PM | Long self-play training runs, staggered so we don't overload shared compute (all) | Short check-in before Thanksgiving; decide which runs to keep |
| 12/1 | 7 PM | Hyperparameter sweeps with 3 seeds each: simulations and exploration constants (Brycen and Cameron), network size and learning rate (Anshul) | Pick final configurations using Elo and the parsimony rule; plan figures and the report's main point |
| 12/5 | 3 PM | Final round-robin tournament, simulation-efficiency and time-per-move experiments (all); draft Results (Anshul) | Review results; draft Discussion, Limitations, and Ethics |
| 12/8 | 7 PM | Draft Abstract, Background, and Conclusion (Brycen); clean up repo, README, and figures (Cameron) | Edit the full report together |
| 12/11 | Before 11:59 PM | NA | Turn in Final Project |

# Footnotes
<a name="haywardnote"></a>1.[^](#hayward): Hayward, R. B., & Toft, B. (2019). *Hex, Inside and Out: The Full Story*. CRC Press. https://doi.org/10.1201/9780429031960<br>
<a name="galenote"></a>2.[^](#gale): Gale, D. (1979). The Game of Hex and the Brouwer Fixed-Point Theorem. *The American Mathematical Monthly, 86*(10), 818–827. https://doi.org/10.1080/00029890.1979.11994922<br>
<a name="reischnote"></a>3.[^](#reisch): Reisch, S. (1981). Hex ist PSPACE-vollständig. *Acta Informatica, 15*, 167–191. https://doi.org/10.1007/BF00288964<br>
<a name="hendersonnote"></a>4.[^](#henderson): Henderson, P., Arneson, B., & Hayward, R. B. (2009). Solving 8x8 Hex. *Proceedings of the 21st International Joint Conference on Artificial Intelligence (IJCAI)*, 505–509. https://www.ijcai.org/proceedings/2009/91<br>
<a name="pawlewicznote"></a>5.[^](#pawlewicz): Pawlewicz, J., & Hayward, R. B. (2014). Scalable Parallel DFPN Search. *Computers and Games 2013*, Lecture Notes in Computer Science 8427, 138–150. https://doi.org/10.1007/978-3-319-09165-5_12<br>
<a name="anshelevichnote"></a>6.[^](#anshelevich): Anshelevich, V. V. (2002). A Hierarchical Approach to Computer Hex. *Artificial Intelligence, 134*(1–2), 101–120. https://doi.org/10.1016/S0004-3702(01)00154-0<br>
<a name="kocsisnote"></a>7.[^](#kocsis): Kocsis, L., & Szepesvári, C. (2006). Bandit Based Monte-Carlo Planning. *European Conference on Machine Learning (ECML)*, 282–293. https://doi.org/10.1007/11871842_29<br>
<a name="brownenote"></a>8.[^](#browne): Browne, C. B., et al. (2012). A Survey of Monte Carlo Tree Search Methods. *IEEE Transactions on Computational Intelligence and AI in Games, 4*(1), 1–43. https://doi.org/10.1109/TCIAIG.2012.2186810<br>
<a name="arnesonnote"></a>9.[^](#arneson): Arneson, B., Hayward, R. B., & Henderson, P. (2010). Monte Carlo Tree Search in Hex. *IEEE Transactions on Computational Intelligence and AI in Games, 2*(4), 251–258. https://doi.org/10.1109/TCIAIG.2010.2067212<br>
<a name="silver2017note"></a>10.[^](#silver2017): Silver, D., et al. (2017). Mastering the game of Go without human knowledge. *Nature, 550*, 354–359. https://doi.org/10.1038/nature24270<br>
<a name="silver2018note"></a>11.[^](#silver2018): Silver, D., et al. (2018). A general reinforcement learning algorithm that masters chess, shogi, and Go through self-play. *Science, 362*(6419), 1140–1144. https://doi.org/10.1126/science.aar6404<br>
<a name="anthonynote"></a>12.[^](#anthony): Anthony, T., Tian, Z., & Barber, D. (2017). Thinking Fast and Slow with Deep Learning and Tree Search. *Advances in Neural Information Processing Systems (NeurIPS) 30*. https://arxiv.org/abs/1705.08439<br>
<a name="gaonote"></a>13.[^](#gao): Gao, C., Müller, M., & Hayward, R. (2018). Three-Head Neural Network Architecture for Monte Carlo Tree Search. *Proceedings of the 27th International Joint Conference on Artificial Intelligence (IJCAI)*, 3762–3768. https://doi.org/10.24963/ijcai.2018/523<br>
<a name="lanctotnote"></a>14.[^](#lanctot): Lanctot, M., et al. (2019). OpenSpiel: A Framework for Reinforcement Learning in Games. https://arxiv.org/abs/1908.09453<br>
<a name="wilsonnote"></a>15.[^](#wilson): Wilson, E. B. (1927). Probable Inference, the Law of Succession, and Statistical Inference. *Journal of the American Statistical Association, 22*(158), 209–212. https://doi.org/10.1080/01621459.1927.10502953<br>
<a name="elonote"></a>16.[^](#elo): Elo, A. E. (1978). *The Rating of Chessplayers, Past and Present*. Arco Publishing, New York.
